# IGPO｜Information Gain-based Policy Optimization

论文：[arXiv:2510.14967](https://arxiv.org/abs/2510.14967)（ICLR 2026）

**问题**：小 group 下 easy 全对 / hard 全错 → GRPO advantage=0 → 没梯度  
**IGPO**：每轮 search 后 teacher-force GT，用相邻轮 `P(GT)` 差值作过程奖励，再与终局 F1 组成 turn-level advantage

> Runtime → Change runtime type → **T4 GPU**

### 启动方式（二选一）
1. **推荐**：把本机 `igpo-agentic-search.zip` 拖到 Colab 左侧文件栏，然后跑下面格子  
2. 若已 push GitHub：改 `REPO_URL` 后用 git clone

In [ ]:
#@title 1) Bootstrap 代码
import os, zipfile
from pathlib import Path

REPO_URL = os.environ.get("IGPO_REPO", "")  # e.g. https://github.com/<you>/igpo-agentic-search.git
ROOT = Path("/content/igpo-agentic-search")
ZIP_CANDIDATES = [
    Path("/content/igpo-agentic-search.zip"),
    Path("/content/drive/MyDrive/igpo-agentic-search.zip"),
]

if not (ROOT / "igpo").exists():
    zipped = next((p for p in ZIP_CANDIDATES if p.exists()), None)
    if zipped is not None:
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zipped) as z:
            z.extractall(ROOT if any(n.startswith("igpo/") for n in z.namelist()) else "/content")
        # zip 可能解压到 /content/igpo-agentic-search 或扁平到 ROOT
        if not (ROOT / "igpo").exists() and Path("/content/igpo").exists():
            ROOT = Path("/content")
        print("bootstrapped from", zipped)
    elif REPO_URL:
        !git clone {REPO_URL} /content/igpo-agentic-search
        ROOT = Path("/content/igpo-agentic-search")
    else:
        raise FileNotFoundError(
            "请上传 igpo-agentic-search.zip 到 /content/，或设置 REPO_URL"
        )

os.chdir(ROOT if (ROOT / "igpo").exists() else "/content/igpo-agentic-search")
print("cwd=", os.getcwd())
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
#@title 2) 单测：GRPO collapse vs IGPO 仍有信号
!python scripts/smoke_test.py
!pytest -q tests/

In [ ]:
#@title 3) 训练 IGPO（Qwen2.5-0.5B + LoRA）
from igpo.train.trainer import TrainConfig, run_training

cfg = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    algo="igpo",          # 对照可改 "grpo"
    max_steps=8,
    prompts_per_step=2,
    group_size=4,         # 小 group → collapse 更明显
    max_turns=3,
    max_new_tokens=128,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/igpo_colab",
)
history = run_training(cfg)
history[-1]

In [ ]:
#@title 4) 曲线：F1 / collapse / |IG|
import matplotlib.pyplot as plt

steps = [h.step for h in history]
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].plot(steps, [h.mean_f1 for h in history]); axes[0].set_title("mean F1")
axes[1].plot(steps, [h.collapse_rate for h in history]); axes[1].set_title("outcome collapse rate")
axes[2].plot(steps, [h.mean_abs_ig for h in history]); axes[2].set_title("mean |IG|")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout(); plt.show()

### 面试口述
1. **Advantage collapse**：group 内 outcome 全同 → z-score=0  
2. **IG 奖励**：`r_t = P(GT|ctx_t)-P(GT|ctx_{t-1})`，teacher forcing，内生低 cost  
3. **稠密优势**：IG turns + 终局 F1 → separate z-norm → γ 折扣回传  
4. **对比**：相对 MCTS / 外部 RM，更不易 hacking，且每条样本都有梯度信号